# Phase 2 Lab - Backend API And Database

Mục tiêu: hiểu FastAPI app, schema validation, SQLite initialization, và API
contract đầu tiên.

Expected output chính: focused API tests pass; `/health` trả
`{"status": "ok", "service": "shopping-assistant-v3"}`.

Safety: default tests không gọi live services hoặc model APIs.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy focused API tests

Command này kiểm tra `/health`, tạo chat job, validation errors, read job status
và repository side effects qua temp SQLite database.


In [ ]:
run(["uv", "run", "pytest", "tests/test_api.py", "-q", "--tb=short"], timeout=180)


Nếu command timeout trong sandbox, đó có thể là vấn đề `TestClient`/AnyIO đã
được ghi ở Phase 5A review. Trên máy local, implementer report ghi nhận API
tests pass.


## 2. Gọi Python API/schema trực tiếp

Cell này không mở server. Nó import FastAPI handler/schema để bạn thấy shape dữ
liệu Phase 2.


In [ ]:
from backend.api.main import health
from backend.api.schemas import ChatJobRequest

print(health())
request = ChatJobRequest(message="Tìm laptop gaming dưới 800 đô")
print(request.model_dump())


## 3. Xem SQLite tables được định nghĩa

Expected tables: `jobs`, `conversations`, `messages`, `agent_runs`, `products`,
`price_estimates`.


In [ ]:
from backend.database.schema import Base

for table_name in sorted(Base.metadata.tables):
    print(table_name)


## 4. Cách đọc kết quả

- API tests pass: Phase 2 API/database contract ổn.
- Health dict đúng: app boot được.
- Request schema có default `source='All'` và `max_results_per_source=5`.
- Trên current Phase 5A code, worker có thể xử lý job sau khi tạo; đây là khác
  với historical Phase 2, nơi job chỉ dừng ở `pending`.
